# Glaucoma detection app (Colab GPU version)
Runs the full app with **MedGemma** on a Colab GPU. The public Streamlit Cloud version uses a Gemini/Groq fallback instead.

Required Colab secrets: `HF_TOKEN`, `NGROK_AUTH_TOKEN` (optional: `GEMINI_API_KEY`, `GROQ_API_KEY`).

In [ ]:
!git clone https://github.com/YOUR_GITHUB_USERNAME/glaucoma-app.git
%cd glaucoma-app

In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt

In [ ]:
import os, subprocess, time
from google.colab import userdata
from pyngrok import ngrok

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

# --- Hugging Face token (required for MedGemma) ---
hf = secret("HF_TOKEN")
if not hf:
    raise RuntimeError(
        "HF_TOKEN secret not found. Add it in Colab: left sidebar > key icon > Add new secret "
        "(name: HF_TOKEN) and turn on 'Notebook access'. The token must come from an account that "
        "accepted the license at https://huggingface.co/google/medgemma-1.5-4b-it"
    )
os.environ["HF_TOKEN"] = hf
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf

# --- Optional backup keys (only used if ALLOW_FALLBACK = True in app.py) ---
if secret("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY")
if secret("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = secret("GROQ_API_KEY")

os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Check GPU without importing torch in the notebook ---
print(subprocess.run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv",
                     shell=True, capture_output=True, text=True).stdout or "NO GPU - MedGemma will not run")

# --- Stop any previous run ---
subprocess.run("pkill -9 -f streamlit", shell=True)
ngrok.kill()
time.sleep(2)

# --- Ngrok ---
try:
    ngrok.set_auth_token(secret("NGROK_AUTH_TOKEN"))
except Exception as e:
    if "Authtoken already set in configuration file" in str(e):
        print("Ngrok auth token is already configured.")
    else:
        raise

# --- Start Streamlit (inherits the environment variables above) ---
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT,
    env=os.environ.copy())
time.sleep(8)
print("Open this link:", ngrok.connect(8501).public_url)